# Chapter 2 - Working with Text

## Tokenizing Text

Load a text file for use in this chapter

In [1]:
with open("../../ch02/01_main-chapter-code/the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print("Total number of characters:", len(raw_text))
print(raw_text[:99])

Total number of characters: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


A simplistic tokenizer would just be splitting the text on words.

In [2]:
import re

text = "Hello, world. This, is a test."
result = re.split(r"(\s)", text)
print(result)

['Hello,', ' ', 'world.', ' ', 'This,', ' ', 'is', ' ', 'a', ' ', 'test.']


This sort of works, but notice that punctuation is attached to the words. Let's expand the regex to include punctuation,

In [3]:
result = re.split(r"([,.]|\s)", text)
print(result)

['Hello', ',', '', ' ', 'world', '.', '', ' ', 'This', ',', '', ' ', 'is', ' ', 'a', ' ', 'test', '.', '']


This is working but limited in punctuation and includes whitespace. Whitespace is useful for coding, but less so for straight text, so let's remove it and add in additional punctuation.

In [4]:
result = re.split(r'([,.:;?_!"()\']|--|\s)', raw_text)
result = [item.strip() for item in result if item.strip()]
print("Tokens:", len(result))
print(result[:30])

Tokens: 4690
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']


## Converting tokens to token IDs
The next thing we need to do is convert the tokens into numerical values. We do this by sorting a unique list and assigning a value to each item in the vocabulary.

In [5]:
all_words = sorted(set(result))
vocab_size = len(all_words)
print("Vocab:", vocab_size)

Vocab: 1130


Now that we have the size, we can create the vocabulary with IDs

In [6]:
vocab = {token: integer for integer, token in enumerate(all_words)}

# Display the first 20
for i, item in enumerate(vocab.items()):
    print(item)
    if i >= 20:
        break

# Print the last Id
print("Last ID:", vocab[all_words[-1]])

('!', 0)
('"', 1)
("'", 2)
('(', 3)
(')', 4)
(',', 5)
('--', 6)
('.', 7)
(':', 8)
(';', 9)
('?', 10)
('A', 11)
('Ah', 12)
('Among', 13)
('And', 14)
('Are', 15)
('Arrt', 16)
('As', 17)
('At', 18)
('Be', 19)
('Begin', 20)
Last ID: 1129


Now we can use the IDs we created to convert the source text into token IDs, but we will also need to be able to turn IDs back into text, so first we'll create a Python class to go both ways.

In [7]:
class SimpleTokenizerV1:
    def __init__(self, vocab) -> None:
        self.str_to_int = vocab
        self.int_to_str = {i: s for s, i in vocab.items()}

    def encode(self, text: str) -> list[int]:
        prepocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        prepocessed = [item.strip() for item in prepocessed if item.strip()]
        ids = [self.str_to_int[s] for s in prepocessed]
        return ids

    def decode(self, ids: list[int]) -> str:
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?_!"()\'])', r"\1", text)
        return text

We can now use this class to tokenize a passage from the text.

In [8]:
tokenizer = SimpleTokenizerV1(vocab)
text = """"It's the last he painted, you know,"
  Mrs. Gisburn said with pardonable pride."""
ids = tokenizer.encode(text)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


But because the IDs are only associated with the one passage we gave it, new text fails. We'll deal with that next.

## Adding special context tokens

We'll add two new tokens, first `<|unk|>` for unknown tokens and `<|endoftext|>` to separate unrelated texts.

In [12]:
all_tokens = sorted(set(result))
vocab = {token: integer for integer, token in enumerate(all_tokens)}


class SimpleTokenizerV2:
    ENDOFTEXT_TOKEN = "<|endoftext|>"
    UNK_TOKEN = "<|unk|>"
    SPECIAL_TOKENS = [ENDOFTEXT_TOKEN, UNK_TOKEN]

    def __init__(self, vocab: dict[str, int]) -> None:
        # Copy so we don't mutate the caller's vocabulary, then append any
        # special tokens it is missing, giving each the next free ID.
        self.str_to_int = dict(vocab)
        for token in self.SPECIAL_TOKENS:
            if token not in self.str_to_int:
                self.str_to_int[token] = len(self.str_to_int)
        self.int_to_str = {i: s for s, i in self.str_to_int.items()}

    def encode(self, text: str) -> list[int]:
        prepocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        prepocessed = [item.strip() for item in prepocessed if item.strip()]
        prepocessed = [item if item in self.str_to_int else self.UNK_TOKEN for item in prepocessed]
        ids = [self.str_to_int[s] for s in prepocessed]
        return ids

    def decode(self, ids: list[int]) -> str:
        text = " ".join([self.int_to_str[i] for i in ids])
        text = re.sub(r'\s+([,.:;?_!"()\'])', r"\1", text)
        return text


text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."
text = " ".join((text1, SimpleTokenizerV2.ENDOFTEXT_TOKEN, text2))
print(text)

tokenizer = SimpleTokenizerV2(vocab)
encoded = tokenizer.encode(text)
print(encoded)
print(tokenizer.decode(encoded))

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.
[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]
<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.
